# 用 Sciverse 做科学问答的 Citation Grounding

> 为 LLM 回答的每一句话找到可验证的文献来源，消除幻觉

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 拆分草稿为独立论点

将 LLM 生成的回答拆分为可独立验证的句子


In [ ]:
def split_claims(draft: str) -> list[str]:
    """将草稿拆分为独立论点句子"""
    sentences = [s.strip() for s in draft.split("。") if s.strip()]
    claims = [s for s in sentences if len(s) > 10]
    return claims

draft = "mRNA 疫苗使用可电离脂质纳米颗粒(iLNP)包裹 mRNA。其中 MC3 是最广泛使用的可电离脂质。LNP 的粒径通常在 80-100nm。"
claims = split_claims(draft)
print(f"Split into {len(claims)} claims:")
for c in claims:
    print(f"  - {c}")

## Step 2: 逐句检索证据

对每个论点调用 agentic-search 查找支持文献


In [ ]:
import httpx

async def verify_claim(claim: str):
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            "https://api.sciverse.space/agentic-search",
            headers={"Authorization": "Bearer sv-..."},
            json={"query": claim, "top_k": 5}
        )
        hits = resp.json()["hits"]
        evidence = [h for h in hits if h["score"] >= 0.7]
        return {
            "claim": claim,
            "verified": len(evidence) > 0,
            "evidence": evidence[:2] if evidence else []
        }

results = []
for claim in claims:
    result = await verify_claim(claim)
    results.append(result)
    status = "✅" if result["verified"] else "❌"
    print(f"{status} {claim[:40]}...")

## Step 3: 生成带引用的最终回答

将验证结果组装为带 citation 的最终输出


In [ ]:
def build_grounded_answer(results: list) -> dict:
    citations = []
    grounded_parts = []
    unverified = []

    for r in results:
        if r["verified"]:
            cite_id = len(citations) + 1
            citations.append({
                "id": cite_id,
                "doc_id": r["evidence"][0]["doc_id"],
                "title": r["evidence"][0]["title"],
                "verified": True
            })
            grounded_parts.append(f"{r['claim']} [{cite_id}]")
        else:
            grounded_parts.append(f"{r['claim']} [unverified]")
            unverified.append(r["claim"])

    return {
        "grounded_answer": "。".join(grounded_parts) + "。",
        "citations": citations,
        "unverified_claims": unverified
    }

final = build_grounded_answer(results)
print(final["grounded_answer"])
print(f"\nCitations: {len(final['citations'])}")
print(f"Unverified: {len(final['unverified_claims'])}")

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
